In [1]:
import pandas as pd
from validation_structure import normaliser, valider, PAYS_COUVERTS
import re
import requests
import time
from datetime import datetime
import logging

In [2]:
df_tva = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)

In [3]:
df_tva["numero_tva"] = df_tva["numero_tva"].map(normaliser)

df_tva[["tva_valide", "tva_validite"]] = df_tva["numero_tva"].map(valider).tolist()

df_tva_filtre_1 = df_tva[(df_tva["tva_validite"] != "vide") & (df_tva["tva_validite"] != "trop_court") & (df_tva["tva_validite"] != "cle_invalide")]

df_tva_filtre_1["doublon_tva"] = df_tva_filtre_1["numero_tva"].duplicated()

df_tva_filtre_2 = df_tva_filtre_1[df_tva_filtre_1["doublon_tva"] == False][["id", "raison_sociale", "pays_declare", "numero_tva", "date_saisie", "source_saisie"]]

In [4]:
df_echantillon = df_tva_filtre_2.sample(n=200)

In [5]:
df_echantillon

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
8586,8587,Orion Distribution BV,LU,LU78513566,2024-05-08,reprise_erp
4778,4779,Prima Industries GmbH,PT,PT643209000,2025-02-28,crm
3044,3045,Vela Transports SpA,LU,LU62559211,2025-09-16,saisie_manuelle
6643,6644,Delta Consulting GmbH,FR,FR58822044525,2025-07-01,portail_client
2552,2553,Vertex Foods NV,BE,BE1576214564,2024-02-17,reprise_erp
...,...,...,...,...,...,...
1280,1281,Vertex Logistics GmbH,BE,0176829911,2025-02-10,portail_client
6626,6627,Solaris Foods Lda,PL,PL6373560924,2024-06-23,portail_client
4181,4182,Corvus Industries SAS,PL,PL4470697271,2024-11-16,portail_client
3796,3797,Solaris Materials GmbH,PT,PT701713585,2024-05-02,portail_client


In [6]:
def deprefixe(s: str) -> str:
    return re.sub(r"[A-Za-z]", "", s[:2]) + s[2:]

In [7]:
df_echantillon["tva_hors_prefixe"] = df_echantillon["numero_tva"].map(deprefixe)

df_echantillon

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_hors_prefixe
8586,8587,Orion Distribution BV,LU,LU78513566,2024-05-08,reprise_erp,78513566
4778,4779,Prima Industries GmbH,PT,PT643209000,2025-02-28,crm,643209000
3044,3045,Vela Transports SpA,LU,LU62559211,2025-09-16,saisie_manuelle,62559211
6643,6644,Delta Consulting GmbH,FR,FR58822044525,2025-07-01,portail_client,58822044525
2552,2553,Vertex Foods NV,BE,BE1576214564,2024-02-17,reprise_erp,1576214564
...,...,...,...,...,...,...,...
1280,1281,Vertex Logistics GmbH,BE,0176829911,2025-02-10,portail_client,0176829911
6626,6627,Solaris Foods Lda,PL,PL6373560924,2024-06-23,portail_client,6373560924
4181,4182,Corvus Industries SAS,PL,PL4470697271,2024-11-16,portail_client,4470697271
3796,3797,Solaris Materials GmbH,PT,PT701713585,2024-05-02,portail_client,701713585


In [8]:
for row in df_echantillon.itertuples():
    if row.pays_declare == "FR":
        print(f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{row.pays_declare}/vat/{row.tva_hors_prefixe}")

https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/58822044525
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/20764950183
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/96579633519
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/53209351751
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/66597985424
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/31485729123
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/90932820799
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/93727137344
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/81118731709
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/91527985350
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/09807475238
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/31342302498
https://ec.europa.eu/taxation_customs/vies/rest-api/ms/FR/vat/52203860807
https://ec.europa.eu/taxation_customs/

In [9]:
logging.basicConfig(
    filename="journal.log",
    filemode="a",  # 'a' pour ajouter à la suite, 'w' pour écraser
    encoding="utf-8",
    level=logging.INFO,  # DEBUG, INFO, WARNING, ERROR, CRITICAL
    format="%(asctime)s - %(levelname)s - %(message)s",
)

In [10]:
#logs = []
for row in df_echantillon.itertuples():
    try:
        debut = datetime.now()
        r = requests.get(f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{row.pays_declare}/vat/{row.tva_hors_prefixe}", timeout=2)
        #time.sleep(5)
        fin = datetime.now()
        #logs.append({"status_code": r.status_code, "text": r.text, "delai": fin - debut})
        logging.info(f"Démarrage row {row.id} : {debut}")
        if r.status_code == 200:
            logging.info(f"Status_code : {r.status_code} : OK")
        else:
            logging.error(f"Status_code : {r.status_code} : {r.text}")
        logging.info(r.text)
        logging.warning(f"Delai: {fin - debut}")
        #display(logs[-1])
    except requests.exceptions.Timeout:
        logging.info(f"Démarrage row {row.id} : {debut}")
        logging.error(f"ERREUR TIMEOUT : Delai: {fin - debut}")